# MiniGPT — Treinamento no Google Colab

Notebook completo pra treinar o MiniGPT usando a GPU gratuita do Colab.

**Pipeline completo:**
1. Pré-treinamento (corpus sintético ou externo)
2. SFT (Supervised Fine-Tuning) — opcional
3. DPO (Direct Preference Optimization) — opcional

**Novidades:**
- BPE Tokenizer (3-5x mais eficiente que char-level)
- RoPE (Rotary Position Embedding)
- Flash Attention (PyTorch 2.0+)
- Top-p, Repetition Penalty, Typical Sampling, Beam Search
- Gradient Accumulation, Validação, Early Stopping

**Como usar:**
1. Vá em `Ambiente de execução > Alterar o tipo de ambiente de execução` e selecione **T4 GPU**
2. Execute cada célula em ordem (Shift+Enter)
3. No final, baixe os arquivos de output

In [ ]:
#@title 1. Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print(f'Flash Attention: {hasattr(torch.nn.functional, "scaled_dot_product_attention")}')
else:
    print('⚠️ GPU não detectada! Vá em Ambiente de execução > Alterar tipo > T4 GPU')

In [ ]:
#@title 2. Criar todos os arquivos do projeto
%%bash
mkdir -p minigpt/data minigpt/output

cat > minigpt/config.py << 'PYEOF'
"""config.py — Central de hiperparâmetros do MiniGPT"""
from dataclasses import dataclass

@dataclass
class GPTConfig:
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    context_len: int = 256
    dropout: float = 0.3
    batch_size: int = 64
    learning_rate: float = 3e-4
    weight_decay: float = 0.3
    max_epochs: int = 30
    beta1: float = 0.9
    beta2: float = 0.95
    max_grad_norm: float = 1.0
    warmup_steps: int = 200
    min_lr: float = 1e-5
    gradient_accum_steps: int = 1
    val_split: float = 0.1
    patience: int = 8
    top_p: float = 0.9
    repetition_penalty: float = 1.2
    typical_mass: float = 1.0
    tokenizer_type: str = "bpe"
    bpe_vocab_size: int = 500
    use_rope: bool = True
    beam_width: int = 1
    sft_lr: float = 1e-5
    sft_epochs: int = 5
    dpo_beta: float = 0.1
    dpo_lr: float = 1e-6
    dpo_epochs: int = 3
    vocab_size: int = 0

    @property
    def head_dim(self) -> int:
        assert self.d_model % self.n_heads == 0, (
            f"d_model ({self.d_model}) deve ser divisível por n_heads ({self.n_heads})"
        )
        return self.d_model // self.n_heads
PYEOF

echo "✓ config.py"

In [ ]:
#@title 3. Criar tokenizer.py
%%bash
cat > minigpt/tokenizer.py << 'PYEOF'
"""tokenizer.py — CharTokenizer + BPETokenizer"""
import json, re
from collections import Counter
from pathlib import Path

TOKENS_ESPECIAIS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]

class CharTokenizer:
    def __init__(self):
        self.char_to_id = {}
        self.id_to_char = {}
        self.vocab_size = 0
    def treinar(self, texto):
        chars_unicos = sorted(set(texto))
        self.char_to_id = {}
        self.id_to_char = {}
        for i, token in enumerate(TOKENS_ESPECIAIS):
            self.char_to_id[token] = i
            self.id_to_char[i] = token
        for i, ch in enumerate(chars_unicos, start=len(TOKENS_ESPECIAIS)):
            self.char_to_id[ch] = i
            self.id_to_char[i] = ch
        self.vocab_size = len(self.char_to_id)
    def codificar(self, texto):
        return [self.char_to_id.get(ch, 1) for ch in texto]
    def decodificar(self, ids):
        return "".join(self.id_to_char.get(id_, "") for id_ in ids if self.id_to_char.get(id_, "") not in TOKENS_ESPECIAIS)
    def salvar(self, caminho):
        Path(caminho).write_text(json.dumps({"tipo": "char", "char_to_id": self.char_to_id, "id_to_char": {str(k): v for k, v in self.id_to_char.items()}, "vocab_size": self.vocab_size}, ensure_ascii=False, indent=2))
    @classmethod
    def carregar(cls, caminho):
        dados = json.loads(Path(caminho).read_text())
        tok = cls()
        tok.char_to_id = dados["char_to_id"]
        tok.id_to_char = {int(k): v for k, v in dados["id_to_char"].items()}
        tok.vocab_size = dados["vocab_size"]
        return tok
    def __repr__(self):
        return f"CharTokenizer(vocab_size={self.vocab_size})"

class BPETokenizer:
    def __init__(self):
        self.merges = []
        self.vocab = {}
        self.id_to_token = {}
        self.vocab_size = 0
    def treinar(self, texto, vocab_size=500):
        chunks = re.findall(r'\S+|\s+', texto)
        freq_chunks = Counter(chunks)
        chunk_splits = {c: list(c) for c in freq_chunks}
        todos_chars = sorted(set(c for pieces in chunk_splits.values() for c in pieces))
        base_tokens = list(TOKENS_ESPECIAIS) + todos_chars
        self.vocab = {t: i for i, t in enumerate(base_tokens)}
        self.id_to_token = {i: t for i, t in enumerate(base_tokens)}
        self.merges = []
        while len(self.vocab) < vocab_size:
            pair_freq = Counter()
            for chunk, count in freq_chunks.items():
                pieces = chunk_splits[chunk]
                for i in range(len(pieces) - 1):
                    pair_freq[(pieces[i], pieces[i+1])] += count
            if not pair_freq:
                break
            melhor_par = pair_freq.most_common(1)[0][0]
            self.merges.append(melhor_par)
            novo_token = melhor_par[0] + melhor_par[1]
            self.vocab[novo_token] = len(self.vocab)
            self.id_to_token[len(self.id_to_token)] = novo_token
            for chunk in chunk_splits:
                pieces = chunk_splits[chunk]
                new_pieces, i = [], 0
                while i < len(pieces):
                    if i < len(pieces) - 1 and pieces[i] == melhor_par[0] and pieces[i+1] == melhor_par[1]:
                        new_pieces.append(novo_token)
                        i += 2
                    else:
                        new_pieces.append(pieces[i])
                        i += 1
                chunk_splits[chunk] = new_pieces
        self.vocab_size = len(self.vocab)
    def codificar(self, texto):
        if not texto:
            return []
        chunks = re.findall(r'\S+|\s+', texto)
        ids = []
        unk_id = self.vocab.get("<UNK>", 1)
        for chunk in chunks:
            pieces = list(chunk)
            for merge in self.merges:
                new_pieces, i = [], 0
                while i < len(pieces):
                    if i < len(pieces) - 1 and pieces[i] == merge[0] and pieces[i+1] == merge[1]:
                        new_pieces.append(merge[0] + merge[1])
                        i += 2
                    else:
                        new_pieces.append(pieces[i])
                        i += 1
                pieces = new_pieces
            for piece in pieces:
                ids.append(self.vocab.get(piece, unk_id))
        return ids
    def decodificar(self, ids):
        return "".join(self.id_to_token.get(id_, "") for id_ in ids if self.id_to_token.get(id_, "") not in TOKENS_ESPECIAIS)
    def salvar(self, caminho):
        Path(caminho).write_text(json.dumps({"tipo": "bpe", "merges": [[a, b] for a, b in self.merges], "vocab": self.vocab, "id_to_token": {str(k): v for k, v in self.id_to_token.items()}, "vocab_size": self.vocab_size}, ensure_ascii=False, indent=2))
    @classmethod
    def carregar(cls, caminho):
        dados = json.loads(Path(caminho).read_text())
        tok = cls()
        tok.merges = [tuple(m) for m in dados["merges"]]
        tok.vocab = dados["vocab"]
        tok.id_to_token = {int(k): v for k, v in dados["id_to_token"].items()}
        tok.vocab_size = dados["vocab_size"]
        return tok
    def __repr__(self):
        return f"BPETokenizer(vocab_size={self.vocab_size}, merges={len(self.merges)})"

def criar_tokenizer(tipo="bpe"):
    return BPETokenizer() if tipo == "bpe" else CharTokenizer()

def carregar_tokenizer(caminho):
    dados = json.loads(Path(caminho).read_text())
    return BPETokenizer.carregar(caminho) if dados.get("tipo") == "bpe" else CharTokenizer.carregar(caminho)
PYEOF

echo "✓ tokenizer.py"

In [ ]:
#@title 4. Criar model.py (com RoPE + Flash Attention)
%%bash
cat > minigpt/model.py << 'PYEOF'
"""model.py — GPT com RoPE + Flash Attention"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import GPTConfig

def _rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq_len)
    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())
    def forward(self, x, offset=0):
        seq_len = x.shape[-2]
        cos = self.cos_cached[offset : offset + seq_len]
        sin = self.sin_cached[offset : offset + seq_len]
        while cos.dim() < x.dim():
            cos = cos.unsqueeze(0)
            sin = sin.unsqueeze(0)
        return x * cos + _rotate_half(x) * sin

class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gain = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.gain * (x - mean) / torch.sqrt(var + self.eps) + self.bias

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.n_heads = config.n_heads
        self.head_dim = config.head_dim
        self.d_model = config.d_model
        self.use_rope = config.use_rope
        self.W_qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.W_out = nn.Linear(config.d_model, config.d_model)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        if config.use_rope:
            self.rotary_emb = RotaryEmbedding(config.head_dim, config.context_len)
        else:
            self.rotary_emb = None
        self._use_flash = hasattr(F, "scaled_dot_product_attention")

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        if self.use_rope and self.rotary_emb is not None:
            q = self.rotary_emb(q)
            k = self.rotary_emb(k)
        if self._use_flash:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.attn_dropout.p if self.training else 0.0)
        else:
            attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            mascara_causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            attn_scores = attn_scores.masked_fill(mascara_causal, float("-inf"))
            attn_probs = F.softmax(attn_scores, dim=-1)
            attn_probs = self.attn_dropout(attn_probs)
            out = attn_probs @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_out(out)
        return self.resid_dropout(out)

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(config.d_model, 4 * config.d_model), nn.GELU(), nn.Linear(4 * config.d_model, config.d_model), nn.Dropout(config.dropout))
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.d_model)
        self.attn = MultiHeadSelfAttention(config)
        self.ln2 = LayerNorm(config.d_model)
        self.ffn = FeedForward(config)
    def forward(self, x):
        return x + self.ffn(self.ln2(x + self.attn(self.ln1(x))))

class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        if config.use_rope:
            self.position_embedding = None
        else:
            self.position_embedding = nn.Embedding(config.context_len, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f = LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self._init_weights()

    def _init_weights(self):
        n_layers = self.config.n_layers
        for nome, modulo in self.named_modules():
            if isinstance(modulo, nn.Linear):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
                if modulo.bias is not None:
                    nn.init.zeros_(modulo.bias)
            elif isinstance(modulo, nn.Embedding):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
            if nome.endswith(("ffn.net.0", "W_qkv", "W_out")):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        if self.position_embedding is not None:
            pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
            x = self.drop(tok_emb + self.position_embedding(pos))
        else:
            x = self.drop(tok_emb)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

    def contar_parametros(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def resumo(self):
        n = self.contar_parametros()
        c = self.config
        lines = [
            f"════════════════ MiniGPT — Resumo do Modelo ════════════════",
            f"  Parâmetros:     {n:,}",
            f"  Vocabulário:    {c.vocab_size}",
            f"  d_model:        {c.d_model}",
            f"  n_heads:        {c.n_heads}",
            f"  head_dim:       {c.head_dim}",
            f"  n_layers:       {c.n_layers}",
            f"  context_len:    {c.context_len}",
            f"  dropout:        {c.dropout}",
            f"  FFN dim:        {4 * c.d_model}",
            f"  RoPE:           {'Sim' if c.use_rope else 'Não'}",
            f"  Flash Attn:     {'Disponível' if hasattr(F, 'scaled_dot_product_attention') else 'Não disponível'}",
            f"════════════════════════════════════════════════════════════",
        ]
        return "\n".join(lines)
PYEOF

echo "✓ model.py"

In [ ]:
#@title 5. Criar train.py, generate.py, sft.py, dpo.py, data/corpus.py
%%bash

cat > minigpt/train.py << 'PYEOF'
"""
train.py — Loop de treinamento do MiniGPT

EXPANDIDO com:
- Gradient accumulation: simula batch maior sem estourar VRAM
- Split de validação: avalia o modelo em dados não vistos
- Early stopping: para o treino se não melhorar por N épocas
- Tokenizador BPE (ou char, configurável)
"""

import json
import math
import time
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data import random_split

from config import GPTConfig
from data.corpus import augmentar_texto, permutar_frases
from model import GPTModel
from tokenizer import CharTokenizer, BPETokenizer, criar_tokenizer, carregar_tokenizer


class TextDataset(Dataset):
    """Dataset que transforma texto tokenizado em pares (input, target)."""

    def __init__(self, tokens: list[int], context_len: int):
        self.tokens = tokens
        self.context_len = context_len

    def __len__(self) -> int:
        return max(0, len(self.tokens) - self.context_len)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        chunk = self.tokens[idx : idx + self.context_len + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y


def get_lr(it: int, config: GPTConfig) -> float:
    """
    Scheduler de learning rate: Warmup + Cosine Decay.

    lr │    ___....───────┐
       │   /              ╲
       │  /                ╲
       │ /                  ╲___....──── min_lr
       └───────────────────────────────→ step
           warmup   cosine decay
    """
    if it < config.warmup_steps:
        return config.learning_rate * (it + 1) / config.warmup_steps

    progress = (it - config.warmup_steps) / max(1, 10000 - config.warmup_steps)
    progress = min(progress, 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return config.min_lr + coeff * (config.learning_rate - config.min_lr)


@torch.no_grad()
def avaliar(
    modelo: GPTModel, dataloader: DataLoader, device: str
) -> float:
    """Avalia o modelo no dataset de validação. Retorna a loss média."""
    modelo.eval()
    total_loss = 0.0
    n_batches = 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        _, loss = modelo(x, y)
        if loss is not None:
            total_loss += loss.item()
            n_batches += 1
    modelo.train()
    return total_loss / max(n_batches, 1)


def treinar(
    config: GPTConfig,
    texto: str,
    saida_dir: str = "output",
    device: str | None = None,
) -> tuple[GPTModel, CharTokenizer | BPETokenizer]:
    """
    Loop de treinamento completo do MiniGPT.

    NOVIDADES:
    - Gradient accumulation (batch efetivo = batch_size * gradient_accum_steps)
    - Split de validação (val_split % dos dados)
    - Early stopping (pára se não melhorar por patience épocas)
    - Suporte a BPE ou char tokenizer
    """
    if device is None:
        if torch.cuda.is_available():
            device = "cuda"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"
    print(f"Dispositivo: {device}")

    saida_path = Path(saida_dir)
    saida_path.mkdir(exist_ok=True)

    # ── Tokenizer ──
    print("Construindo tokenizer...")
    tokenizer = criar_tokenizer(config.tokenizer_type)
    if isinstance(tokenizer, BPETokenizer):
        tokenizer.treinar(texto, vocab_size=config.bpe_vocab_size)
    else:
        tokenizer.treinar(texto)
    print(f"  {tokenizer}")
    config.vocab_size = tokenizer.vocab_size

    # ── Dataset ──
    tokens = tokenizer.codificar(texto)
    print(f"  Corpus: {len(texto):,} caracteres → {len(tokens):,} tokens")
    print(f"  Tokens únicos: {tokenizer.vocab_size}")

    # ── Data augmentation ──
    # Cria variações do corpus para reduzir overfitting
    import random as _random
    _random.seed(42)

    # Permutar frases: embaralha a ordem das frases
    texto_permutado = permutar_frases(texto)
    tokens_permutados = tokenizer.codificar(texto_permutado)

    # Augmentação com dropout de tokens
    texto_augmentado = augmentar_texto(texto, prob_dropout=0.05)
    tokens_augmentados = tokenizer.codificar(texto_augmentado)

    print(f"  Augmentação: +{len(tokens_permutados):,} tokens (permutação) +{len(tokens_augmentados):,} tokens (dropout)")

    # Split treino/validação (apenas nos tokens originais)
    val_size = int(len(tokens) * config.val_split)
    val_tokens = tokens[-val_size:] if val_size > 0 else None

    # Treino = original (sem val) + permutado + augmentado
    train_tokens = tokens[:-val_size] if val_size > 0 else tokens
    train_tokens = train_tokens + tokens_permutados + tokens_augmentados

    train_dataset = TextDataset(train_tokens, config.context_len)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=device != "cpu",
    )

    val_loader = None
    if val_tokens and len(val_tokens) > config.context_len:
        val_dataset = TextDataset(val_tokens, config.context_len)
        val_loader = DataLoader(
            val_dataset,
            batch_size=config.batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=device != "cpu",
        )

    print(f"  Batch size: {config.batch_size}")
    print(f"  Gradient accumulation: {config.gradient_accum_steps}x")
    print(f"  Batch efetivo: {config.batch_size * config.gradient_accum_steps}")
    print(f"  Batches por época: {len(train_loader)}")
    print(f"  Validação: {'Sim' if val_loader else 'Não'}")

    # ── Modelo ──
    modelo = GPTModel(config).to(device)
    print(modelo.resumo())

    # ── Otimizador ──
    param_decay = [p for n, p in modelo.named_parameters() if p.dim() >= 2]
    param_nodecay = [p for n, p in modelo.named_parameters() if p.dim() < 2]

    otimizador = torch.optim.AdamW(
        [
            {"params": param_decay, "weight_decay": config.weight_decay},
            {"params": param_nodecay, "weight_decay": 0.0},
        ],
        lr=config.learning_rate,
        betas=(config.beta1, config.beta2),
    )

    # ── Loop de treino ──
    modelo.train()
    step_global = 0
    melhor_val_loss = float("inf")
    melhor_train_loss = float("inf")
    epochs_sem_melhora = 0
    media_loss = 0.0
    epoca = 0

    total_tokens_treinados = 0
    tempo_inicio = time.time()
    log_epocas = []

    print(f"\nIniciando treino por {config.max_epochs} épocas...")
    print("=" * 70)

    for epoca in range(1, config.max_epochs + 1):
        modelo.train()
        epoca_loss = 0.0
        n_batches = 0
        t0 = time.time()
        otimizador.zero_grad(set_to_none=True)
        accum_count = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)

            # Forward
            logits, loss = modelo(x, y)

            # Normalizar loss pelo accumulation steps
            loss_norm = loss / config.gradient_accum_steps
            loss_norm.backward()

            accum_count += 1
            epoca_loss += loss.item()
            n_batches += 1
            total_tokens_treinados += x.shape[0] * x.shape[1]

            # Backward + update a cada gradient_accum_steps
            if accum_count % config.gradient_accum_steps == 0:
                lr = get_lr(step_global, config)
                for param_group in otimizador.param_groups:
                    param_group["lr"] = lr

                torch.nn.utils.clip_grad_norm_(
                    modelo.parameters(), config.max_grad_norm
                )
                otimizador.step()
                otimizador.zero_grad(set_to_none=True)
                step_global += 1

        # Remanescente de accumulation
        if accum_count % config.gradient_accum_steps != 0:
            lr = get_lr(step_global, config)
            for param_group in otimizador.param_groups:
                param_group["lr"] = lr
            torch.nn.utils.clip_grad_norm_(
                modelo.parameters(), config.max_grad_norm
            )
            otimizador.step()
            otimizador.zero_grad(set_to_none=True)
            step_global += 1

        media_loss = epoca_loss / max(n_batches, 1)
        elapsed = time.time() - t0
        perplexidade = math.exp(min(media_loss, 20))

        lr_atual = get_lr(step_global, config)

        # Validação
        val_str = ""
        val_loss = None
        if val_loader:
            val_loss = avaliar(modelo, val_loader, device)
            val_str = f" │ Val Loss: {val_loss:.4f}"

        print(
            f"Época {epoca:3d}/{config.max_epochs} │ "
            f"Loss: {media_loss:.4f} │ "
            f"PPL: {perplexidade:.2f} │ "
            f"LR: {lr_atual:.2e} │ "
            f"Tempo: {elapsed:.1f}s"
            f"{val_str}"
        )

        log_epocas.append({
            "epoca": epoca,
            "loss": round(media_loss, 6),
            "ppl": round(perplexidade, 2),
            "lr": lr_atual,
            "tempo_seg": round(elapsed, 1),
            **({"val_loss": round(val_loss, 6)} if val_loss is not None else {}),
        })

        # Salvar melhor modelo (baseado em val_loss se houver, senão train_loss)
        loss_monitor = val_loss if val_loss is not None else media_loss
        loss_ref = melhor_val_loss if val_loss is not None else melhor_train_loss

        if loss_monitor < loss_ref:
            if val_loss is not None:
                melhor_val_loss = val_loss
            else:
                melhor_train_loss = media_loss
            epochs_sem_melhora = 0

            torch.save(
                {
                    "modelo": modelo.state_dict(),
                    "config": config,
                    "epoch": epoca,
                    "loss": media_loss,
                    **({"val_loss": val_loss} if val_loss is not None else {}),
                    "total_tokens": total_tokens_treinados,
                    "tempo_total_seg": time.time() - tempo_inicio,
                },
                saida_path / "melhor_modelo.pt",
            )
            tokenizer.salvar(str(saida_path / "tokenizer.json"))
        else:
            epochs_sem_melhora += 1

        # Early stopping
        if config.patience > 0 and epochs_sem_melhora >= config.patience:
            print(f"\nEarly stopping! Sem melhora há {config.patience} épocas.")
            break

    tempo_total = time.time() - tempo_inicio
    tokens_por_segundo = total_tokens_treinados / max(tempo_total, 1)

    # Salvar modelo final
    torch.save(
        {
            "modelo": modelo.state_dict(),
            "config": config,
            "epoch": epoca,
            "loss": media_loss,
            "total_tokens": total_tokens_treinados,
            "tempo_total_seg": tempo_total,
        },
        saida_path / "modelo_final.pt",
    )

    melhor = melhor_val_loss if val_loader else melhor_train_loss
    print("=" * 70)
    print(f"Treino concluído! Melhor loss: {melhor:.4f}")
    print(f"Tempo total: {tempo_total:.1f}s ({tempo_total/60:.1f} min)")
    print(f"Tokens treinados: {total_tokens_treinados:,}")
    print(f"Throughput: {tokens_por_segundo:,.0f} tokens/s")
    print(f"Modelo salvo em: {saida_path / 'melhor_modelo.pt'}")
    print(f"Tokenizer salvo em: {saida_path / 'tokenizer.json'}")

    # Salvar log
    log_treino = {
        "config": {
            "d_model": config.d_model,
            "n_heads": config.n_heads,
            "n_layers": config.n_layers,
            "context_len": config.context_len,
            "dropout": config.dropout,
            "batch_size": config.batch_size,
            "gradient_accum_steps": config.gradient_accum_steps,
            "learning_rate": config.learning_rate,
            "weight_decay": config.weight_decay,
            "max_epochs": config.max_epochs,
            "warmup_steps": config.warmup_steps,
            "min_lr": config.min_lr,
            "max_grad_norm": config.max_grad_norm,
            "vocab_size": config.vocab_size,
            "tokenizer_type": config.tokenizer_type,
            "use_rope": config.use_rope,
        },
        "device": device,
        "n_params": modelo.contar_parametros(),
        "total_tokens_treinados": total_tokens_treinados,
        "tempo_total_seg": round(tempo_total, 1),
        "tempo_total_min": round(tempo_total / 60, 1),
        "tokens_por_segundo": round(tokens_por_segundo, 1),
        "melhor_loss": round(melhor, 6),
        "epocas": log_epocas,
    }
    log_path = saida_path / "log_treino.json"
    log_path.write_text(json.dumps(log_treino, indent=2, ensure_ascii=False))
    print(f"Log salvo em: {log_path}")

    return modelo, tokenizer
PYEOF

cat > minigpt/generate.py << 'PYEOF'
"""generate.py — Geração com top-p, repetition penalty, typical sampling, beam search"""
import torch
import torch.nn.functional as F
from model import GPTModel
from tokenizer import CharTokenizer, BPETokenizer

def apply_top_k(logits, top_k):
    if top_k is None or top_k <= 0:
        return logits
    kth_vals = torch.topk(logits, min(top_k, logits.size(-1)))[0][:, -1:]
    return logits.masked_fill(logits < kth_vals, float("-inf"))

def apply_top_p(logits, top_p):
    if top_p >= 1.0:
        return logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = False
    indices_to_remove = torch.zeros_like(logits, dtype=torch.bool)
    indices_to_remove.scatter_(-1, sorted_indices, sorted_indices_to_remove)
    return logits.masked_fill(indices_to_remove, float("-inf"))

def apply_repetition_penalty(logits, tokens_gerados, penalty):
    if penalty == 1.0 or not tokens_gerados:
        return logits
    for token_id in set(tokens_gerados):
        if token_id < logits.size(-1):
            if logits[0, token_id] > 0:
                logits[0, token_id] /= penalty
            else:
                logits[0, token_id] *= penalty
    return logits

@torch.no_grad()
def gerar(modelo, tokenizer, prompt, max_tokens=200, temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.2, typical_mass=1.0, device="cpu"):
    modelo.eval()
    tokens = tokenizer.codificar(prompt)
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    eos_id = tokenizer.vocab.get("<EOS>", -1) if isinstance(tokenizer, BPETokenizer) else tokenizer.char_to_id.get("<EOS>", -1)
    tokens_gerados = list(tokens)
    for _ in range(max_tokens):
        idx_cond = idx[:, -modelo.config.context_len :]
        logits, _ = modelo(idx_cond)
        logits = logits[:, -1, :]
        logits = logits / max(temperature, 1e-8)
        logits = apply_repetition_penalty(logits, tokens_gerados, repetition_penalty)
        logits = apply_top_k(logits, top_k)
        logits = apply_top_p(logits, top_p)
        probs = F.softmax(logits, dim=-1)
        proximo_token = torch.multinomial(probs, num_samples=1)
        if proximo_token.item() == eos_id:
            break
        tokens_gerados.append(proximo_token.item())
        idx = torch.cat([idx, proximo_token], dim=1)
    return tokenizer.decodificar(idx[0].tolist())

def gerar_variacoes(modelo, tokenizer, prompt, n_variacoes=3, max_tokens=200, temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.2, device="cpu"):
    return [gerar(modelo, tokenizer, prompt, max_tokens, temperature, top_k, top_p, repetition_penalty, device=device) for _ in range(n_variacoes)]

@torch.no_grad()
def gerar_beam_search(modelo, tokenizer, prompt, max_tokens=100, beam_width=5, temperature=1.0, length_penalty=0.6, device="cpu"):
    modelo.eval()
    tokens = tokenizer.codificar(prompt)
    eos_id = tokenizer.vocab.get("<EOS>", -1) if isinstance(tokenizer, BPETokenizer) else tokenizer.char_to_id.get("<EOS>", -1)
    beams = [(0.0, tokens)]
    for _ in range(max_tokens):
        all_candidates = []
        for score, seq in beams:
            idx = torch.tensor([seq[-modelo.config.context_len:]], dtype=torch.long, device=device)
            logits, _ = modelo(idx)
            log_probs = F.log_softmax(logits[:, -1, :] / max(temperature, 1e-8), dim=-1)
            top_probs, top_indices = torch.topk(log_probs[0], beam_width)
            for i in range(beam_width):
                all_candidates.append((score + top_probs[i].item(), seq + [top_indices[i].item()]))
        all_candidates.sort(key=lambda x: x[0] / (len(x[1]) ** length_penalty), reverse=True)
        beams = all_candidates[:beam_width]
        if all(seq[-1] == eos_id for _, seq in beams):
            break
    best_score, best_seq = max(beams, key=lambda x: x[0] / (len(x[1]) ** length_penalty))
    return tokenizer.decodificar(best_seq)
PYEOF

cat > minigpt/sft.py << 'PYEOF'
"""
sft.py — Supervised Fine-Tuning

SFT transforma um modelo pré-treinado (que só prevê próximo token)
num modelo de instrução que segue comandos.

COMO FUNCIONA:
- Formato: "Instrução: {pergunta} Resposta: {resposta}"
- A loss é computada APENAS nos tokens da resposta
- Tokens da instrução recebem target=-100 (ignorados pelo cross_entropy)
- Learning rate menor que o pré-treinamento (1e-5 vs 3e-4)

INTUIÇÃO:
- Pré-treinamento: modelo aprende a LINGUAGEM (português)
- SFT: modelo aprende a SEGUIR INSTRUÇÕES (Q&A, comandos)
"""

import json
import math
import time
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

from config import GPTConfig
from model import GPTModel
from tokenizer import CharTokenizer, BPETokenizer


SEPARATOR = " Resposta: "
IGNORE_INDEX = -100


class SFTDataset(Dataset):
    """
    Dataset para Supervised Fine-Tuning.

    Cada amostra é formatada como:
        Instrução: {instruction} Resposta: {response}

    Targets da instrução são -100 (ignorados).
    Targets da resposta são os tokens reais.
    """

    def __init__(
        self,
        pares: list[tuple[str, str]],
        tokenizer: CharTokenizer | BPETokenizer,
        context_len: int,
    ):
        self.samples = []
        self.context_len = context_len

        for instrucao, resposta in pares:
            # Formato completo
            texto_completo = f"Instrução: {instrucao}{SEPARATOR}{resposta}"
            tokens_completo = tokenizer.codificar(texto_completo)

            # Parte da instrução (para mascarar)
            texto_inst = f"Instrução: {instrucao}{SEPARATOR}"
            tokens_inst = tokenizer.codificar(texto_inst)
            inst_len = len(tokens_inst)

            if len(tokens_completo) > context_len:
                continue

            # Input: tokens_completo, Target: shift right, com mask
            input_ids = tokens_completo
            targets = tokens_completo[1:] + [0]
            # Máscara: ignorar tokens da instrução
            targets = [-100] * min(inst_len, len(targets)) + targets[inst_len:]
            targets = targets[: len(input_ids)]

            if len(input_ids) > 0 and len(targets) > 0:
                self.samples.append(
                    (
                        input_ids[:context_len],
                        targets[:context_len],
                    )
                )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        input_ids, targets = self.samples[idx]
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(targets, dtype=torch.long),
        )


def _sft_collate(
    batch: list[tuple[torch.Tensor, torch.Tensor]],
) -> tuple[torch.Tensor, torch.Tensor]:
    """Pad SFT batches to equal length (shortest power-of-2 that fits)."""
    max_len = max(x.size(0) for x, _ in batch)
    # Arredonda para a próxima potência de 2 (mais eficiente na GPU)
    padded_len = 1
    while padded_len < max_len:
        padded_len *= 2

    input_ids_list, targets_list = [], []
    for x, y in batch:
        pad_len = padded_len - x.size(0)
        input_ids_list.append(
            torch.cat([x, torch.zeros(pad_len, dtype=torch.long)])
        )
        targets_list.append(
            torch.cat([y, torch.full((pad_len,), IGNORE_INDEX, dtype=torch.long)])
        )
    return torch.stack(input_ids_list), torch.stack(targets_list)


def treinar_sft(
    modelo: GPTModel,
    tokenizer: CharTokenizer | BPETokenizer,
    config: GPTConfig,
    dados_sft: list[tuple[str, str]],
    saida_dir: str = "output",
    device: str | None = None,
) -> GPTModel:
    """
    Treina o modelo com Supervised Fine-Tuning.

    Args:
        modelo: modelo pré-treinado
        tokenizer: tokenizer correspondente
        config: hiperparâmetros (usa sft_lr e sft_epochs)
        dados_sft: lista de (instrução, resposta)
        saida_dir: diretório pra salvar
        device: dispositivo
    """
    if device is None:
        if torch.cuda.is_available():
            device = "cuda"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

    print(f"\n{'='*60}")
    print(f"SFT — Supervised Fine-Tuning")
    print(f"Dispositivo: {device}")
    print(f"Dados de treino: {len(dados_sft)} pares instrução-resposta")
    print(f"Learning rate: {config.sft_lr}")
    print(f"Épocas: {config.sft_epochs}")
    print(f"{'='*60}\n")

    saida_path = Path(saida_dir)

    # Dataset
    dataset = SFTDataset(dados_sft, tokenizer, config.context_len)
    if len(dataset) == 0:
        print("ERRO: Nenhuma amostra SFT válida. Aumente context_len ou simplifique os dados.")
        return modelo

    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=device != "cpu",
        collate_fn=_sft_collate,
    )
    print(f"  Amostras SFT válidas: {len(dataset)}")
    print(f"  Batches por época: {len(dataloader)}")

    # Otimizador (LR menor para fine-tuning)
    modelo = modelo.to(device)
    optimizer = torch.optim.AdamW(
        modelo.parameters(),
        lr=config.sft_lr,
        weight_decay=config.weight_decay,
    )

    # Loop de treino
    modelo.train()
    melhor_loss = float("inf")

    for epoca in range(1, config.sft_epochs + 1):
        total_loss = 0.0
        n_batches = 0
        t0 = time.time()

        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            logits, loss = modelo(x, y)
            if loss is None:
                continue

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), config.max_grad_norm)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        media_loss = total_loss / max(n_batches, 1)
        elapsed = time.time() - t0
        print(
            f"SFT Época {epoca:3d}/{config.sft_epochs} │ "
            f"Loss: {media_loss:.4f} │ Tempo: {elapsed:.1f}s"
        )

        # Salvar melhor modelo SFT
        if media_loss < melhor_loss:
            melhor_loss = media_loss
            torch.save(
                {
                    "modelo": modelo.state_dict(),
                    "config": config,
                    "epoch": epoca,
                    "loss": media_loss,
                    "tipo": "sft",
                },
                saida_path / "melhor_modelo_sft.pt",
            )

    modelo.eval()
    print(f"\nSFT concluído! Melhor loss: {melhor_loss:.4f}")
    return modelo
PYEOF

cat > minigpt/dpo.py << 'PYEOF'
"""dpo.py — Direct Preference Optimization"""
import copy, time
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from config import GPTConfig
from model import GPTModel
from tokenizer import CharTokenizer, BPETokenizer

class DPODataset(Dataset):
    def __init__(self, pares, tokenizer, context_len):
        self.samples = []
        self.context_len = context_len
        for prompt, chosen, rejected in pares:
            chosen_ids = tokenizer.codificar(prompt + chosen)
            rejected_ids = tokenizer.codificar(prompt + rejected)
            if len(chosen_ids) > context_len or len(rejected_ids) > context_len:
                continue
            chosen_targets = chosen_ids[1:] + [0]
            rejected_targets = rejected_ids[1:] + [0]
            self.samples.append({"chosen_ids": chosen_ids, "chosen_targets": chosen_targets, "rejected_ids": rejected_ids, "rejected_targets": rejected_targets, "prompt_len": len(tokenizer.codificar(prompt))})
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        mx = max(len(s["chosen_ids"]), len(s["rejected_ids"]), 1)
        def pad(seq, target_len, v=0): return s[seq][:target_len] + [v] * max(0, target_len - len(s[seq][:target_len]))
        return {"chosen_ids": torch.tensor(pad("chosen_ids", mx), dtype=torch.long), "chosen_targets": torch.tensor(pad("chosen_targets", mx, -100), dtype=torch.long), "rejected_ids": torch.tensor(pad("rejected_ids", mx), dtype=torch.long), "rejected_targets": torch.tensor(pad("rejected_targets", mx, -100), dtype=torch.long), "prompt_len": s["prompt_len"]}

def _log_probs(modelo, input_ids, targets):
    logits, _ = modelo(input_ids, targets)
    log_probs = F.log_softmax(logits, dim=-1)
    token_lp = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)
    mask = targets != -100
    return (token_lp * mask).sum(dim=-1)

def treinar_dpo(modelo, tokenizer, config, dados_dpo, saida_dir="output", device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
    print(f"\n{'='*60}\nDPO — Direct Preference Optimization\nDispositivo: {device}\nDados: {len(dados_dpo)} pares\nbeta: {config.dpo_beta}\n{'='*60}\n")
    saida_path = Path(saida_dir)
    modelo_ref = copy.deepcopy(modelo)
    modelo_ref.eval()
    for p in modelo_ref.parameters(): p.requires_grad = False
    modelo = modelo.to(device)
    modelo_ref = modelo_ref.to(device)
    dataset = DPODataset(dados_dpo, tokenizer, config.context_len)
    if len(dataset) == 0:
        print("ERRO: Nenhuma amostra DPO válida. Aumente context_len.")
        return modelo
    dataloader = DataLoader(dataset, batch_size=min(config.batch_size, len(dataset)), shuffle=True, num_workers=0)
    optimizer = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=config.dpo_lr)
    modelo.train()
    beta = config.dpo_beta
    for epoca in range(1, config.dpo_epochs + 1):
        total_loss = 0.0
        n = 0
        t0 = time.time()
        for batch in dataloader:
            ci, ct, ri, rt = batch["chosen_ids"].to(device), batch["chosen_targets"].to(device), batch["rejected_ids"].to(device), batch["rejected_targets"].to(device)
            lc = _log_probs(modelo, ci, ct)
            lr = _log_probs(modelo, ri, rt)
            with torch.no_grad():
                lrc = _log_probs(modelo_ref, ci, ct)
                lrr = _log_probs(modelo_ref, ri, rt)
            loss = -F.logsigmoid(beta * ((lc - lrc) - (lr - lrr))).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), config.max_grad_norm)
            optimizer.step()
            total_loss += loss.item()
            n += 1
        media = total_loss / max(n, 1)
        print(f"DPO Época {epoca:3d}/{config.dpo_epochs} │ Loss: {media:.4f} │ Tempo: {time.time()-t0:.1f}s")
    torch.save({"modelo": modelo.state_dict(), "config": config, "epoch": config.dpo_epochs, "loss": media, "tipo": "dpo"}, saida_path / "melhor_modelo_dpo.pt")
    modelo.eval()
    print(f"\nDPO concluído! Loss final: {media:.4f}")
    return modelo
PYEOF

touch minigpt/data/__init__.py

cat > minigpt/data/corpus.py << 'PYEOF'
"""
data/corpus.py — Corpus sintético em português + augmentação + dados SFT/DPO

EXPANDIDO com:
- Corpus maior e mais variado
- Augmentação de dados (dropout de tokens, permutação de frases)
- Dataset SFT (instrução → resposta)
- Dataset DPO (preferência: resposta boa vs ruim)
- Suporte a corpus externo (arquivo de texto)
"""

import random
from pathlib import Path


# ──────────────────────────────────────────────────────────
# Corpus de pré-treinamento
# ──────────────────────────────────────────────────────────

def gerar_corpus() -> str:
    """Gera corpus sintético em português com padrões variados."""

    sujeitos = [
        "o gato", "a gata", "o cachorro", "a cachorra", "o pássaro",
        "o menino", "a menina", "o professor", "a professora", "o artista",
        "a cozinheira", "o cientista", "a médica", "o engenheiro", "a escritora",
        "o músico", "a rainha", "o rei", "a bailarina", "o pescador",
        "o peixe", "a borboleta", "o urso", "a águia", "o golfinho",
        "a formiga", "o coelho", "a tartaruga", "o leão", "a zebra",
        "o médico", "a enfermeira", "o piloto", "a jornalista", "o agricultor",
        "a advogada", "o arquiteto", "a dentista", "o bombeiro", "a policial",
        "o carteiro", "a psicóloga", "o mecânico", "a veterinária", "o padeiro",
        "a florista", "o rio", "a montanha", "o mar", "a nuvem",
        "o vento", "a chuva", "o sol", "a lua", "a estrela",
        "a flor", "a árvore", "a pedra", "o lago", "a ilha",
    ]

    verbos = [
        "comia", "bebia", "dormia", "corria", "pulava",
        "brincava", "estudava", "trabalhava", "cantava", "dançava",
        "falava", "ouvia", "via", "pensava", "sonhava",
        "escrevia", "lia", "pintava", "cozinhava", "navegava",
        "correu", "pulou", "voou", "nadou", "cantou", "dançou",
        "escreveu", "leu", "pintou", "cozinhou", "navegou", "construiu",
        "descobriu", "ensinou", "aprendeu", "cresceu", "brilhou", "sorriu",
        "chorou", "dormiu", "acordou", "caminhou", "encontrou", "perdeu",
        "ganhou", "amou", "sonhou", "corre", "pula", "voa",
        "nada", "canta", "dança", "escreve", "lê", "pinta",
        "cozinha", "navega", "constrói", "descobre", "ensina", "aprende",
        "cresce", "brilha", "sorri", "chora", "dorme", "acorda",
        "caminha", "encontra", "perde", "ganha", "ama", "sonha",
        "correrá", "pulará", "voará", "nadará", "cantará", "dançará",
        "escreverá", "lerá", "pintará", "cozinhará", "navegará", "construirá",
        "descobrirá", "ensinará", "aprenderá", "crescerá", "brilhará", "sorrirá",
    ]

    objetos = [
        "na praça", "no parque", "na casa", "na escola", "no jardim",
        "na rua", "na praia", "no rio", "na montanha", "na cidade",
        "na floresta", "no campo", "na biblioteca", "no teatro", "na cozinha",
        "no mercado", "na feira", "no hospital", "na igreja", "no estádio",
        "no museu", "no cinema", "no restaurante", "na padaria", "na farmácia",
        "no escritório", "na universidade", "no aeroporto", "na estação", "no porto",
        "na fazenda", "no acampamento", "na caverna", "no deserto", "na neve",
    ]

    complementos = [
        "com alegria", "com cuidado", "com calma", "com atenção",
        "com amor", "com paciência", "com vontade", "sem pressa",
        "com determinação", "com entusiasmo", "com gratidão", "com coragem",
        "com sabedoria", "com esperança", "com fé", "com orgulho",
        "com humildade", "com generosidade", "com justiça", "com verdade",
        "com liberdade", "com delicadeza",
    ]

    adjetivos = [
        "esperto", "curioso", "tranquilo", "corajoso", "gentil", "criativo",
        "alegre", "cuidadoso", "forte", "paciente", "brilhante", "sereno",
        "rápido", "sábio", "generoso", "persistente", "atento", "calmo",
    ]

    gerundios = [
        "estudando", "brincando", "cozinhando", "cantando", "dançando", "lendo",
        "escrevendo", "pintando", "caminhando", "observando", "trabalhando", "viajando",
    ]

    eventos = [
        "choveu", "o telefone tocou", "a campainha soou", "o sol apareceu",
        "a noite chegou", "o vento aumentou", "o trem partiu", "a música começou",
        "a estrela brilhou", "a festa terminou", "o mercado abriu", "a aula começou",
    ]

    condicoes = [
        "chove", "o vento sopra", "a criança lê", "a semente recebe água",
        "a turma coopera", "a família conversa", "o músico pratica", "a cidade descansa",
        "o sol aparece", "a médica escuta", "o atleta treina", "a cientista observa",
    ]

    consequencias = [
        "a planta cresce", "as folhas dançam", "o conhecimento aumenta", "a flor nasce",
        "o trabalho fica leve", "a casa ganha paz", "a canção melhora", "as ruas ficam calmas",
        "o dia fica claro", "o paciente se sente seguro", "o corpo fica forte", "a descoberta acontece",
    ]

    contrastes = [
        "estivesse cansado", "a estrada fosse longa", "o céu estivesse escuro",
        "o problema parecesse difícil", "a chuva caísse forte", "a sala estivesse cheia",
        "o tempo fosse curto", "a montanha fosse alta", "o mar estivesse agitado",
        "a tarefa exigisse esforço",
    ]

    resultados = [
        "o menino continuou", "a viajante chegou feliz", "a estrela apareceu",
        "a equipe encontrou uma solução", "a família cantou na varanda", "a professora explicou com calma",
        "a turma terminou o projeto", "o alpinista alcançou o topo", "o barco voltou ao porto",
        "todos aprenderam algo novo",
    ]

    dialogos_diretos = [
        "Estudem com dedicação.", "Cuidem da natureza.", "Vamos tentar de novo.",
        "Amanhã será melhor.", "A música começa agora.", "Preparem a mesa, por favor.",
        "Observem o céu com atenção.", "A amizade precisa de cuidado.",
    ]

    acoes = [
        "dormir", "viajar", "cozinhar", "estudar", "plantar", "correr",
        "cantar", "pintar", "nadar", "trabalhar", "ler", "ensinar",
    ]

    acoes_passadas = [
        "estudou", "ensinou", "cantou", "dançou", "cozinhou", "viajou",
        "leu", "escreveu", "plantou", "colheu", "observou", "ajudou",
    ]

    frases = []

    # Padrão 1: Sujeito + verbo + lugar
    for s in sujeitos:
        for v in verbos[:45]:
            for o in objetos[:2]:
                frases.append(f"{s.capitalize()} {v} {o}.")

    # Padrão 2: Sujeito + verbo + lugar + complemento
    for s in sujeitos[:20]:
        for v in verbos[10:30]:
            for o in objetos[2:4]:
                for c in complementos[:4]:
                    frases.append(f"{s.capitalize()} {v} {o} {c}.")

    # Padrão 3: Frases negativas
    for s in sujeitos[:30]:
        for v in verbos[:12]:
            frases.append(f"{s.capitalize()} não {v}.")

    # Padrão 4: Perguntas
    for s in sujeitos[:20]:
        for v in verbos[:10]:
            frases.append(f"Será que {s} {v}?")
            frases.append(f"Onde {s} {v}?")

    # Padrão 5: Sujeito + ser + adjetivo
    for i, s in enumerate(sujeitos):
        frases.append(f"{s.capitalize()} é {adjetivos[i % len(adjetivos)]}.")

    # Padrão 6: Sujeito estava + gerúndio + quando + evento
    for i, s in enumerate(sujeitos[:48]):
        frases.append(f"{s.capitalize()} estava {gerundios[i % len(gerundios)]} quando {eventos[i % len(eventos)]}.")

    # Padrão 7: Condição e consequência
    for i, condicao in enumerate(condicoes):
        frases.append(f"Se {condicao}, então {consequencias[i % len(consequencias)]}.")

    # Padrão 8: Contraste e resultado
    for i, contraste in enumerate(contrastes):
        frases.append(f"Embora {contraste}, {resultados[i % len(resultados)]}.")

    # Padrão 9: Diálogo direto em frase narrativa
    for i, fala in enumerate(dialogos_diretos):
        frases.append(f"{sujeitos[i].capitalize()} disse: '{fala}'")

    # Padrão 10: Depois que evento, resultado
    for i, evento in enumerate(eventos):
        frases.append(f"Depois que {evento}, {resultados[i % len(resultados)]}.")

    # Padrão 11: Antes de ação, sujeito ação
    for i, acao in enumerate(acoes):
        frases.append(f"Antes de {acao}, {sujeitos[i + 5]} {verbos[20 + i]}.")

    # Padrão 12: Não apenas ação, mas também ação
    for i, acao in enumerate(acoes_passadas):
        frases.append(f"Não apenas {acao}, mas também {acoes_passadas[(i + 1) % len(acoes_passadas)]}.")

    # Histórias mais longas — storytelling
    historias = [
        "Era uma vez um gato que morava numa casa muito bonita. O gato gostava de dormir no jardim todas as tardes. Um dia, o gato encontrou um pássaro na árvore. O pássaro cantava uma melodia linda. O gato e o pássaro ficaram amigos para sempre. Juntos, exploravam o jardim e brincavam na grama verde. O gato aprendeu que a amizade é o maior tesouro.",
        "A menina estudava todas as noites na biblioteca. Ela lia muitos livros sobre ciência e arte. A menina queria ser uma grande cientista. A professora ajudava a menina com as dúvidas. A menina trabalhava com dedicação e alegria. Um dia, a menina descobriu algo incrível e o mundo celebrou a sua descoberta.",
        "O cachorro corria no parque todas as manhãs. Ele gostava de brincar com a bola. O cachorro era muito feliz. A dona do cachorro cuidava dele com muito amor. Juntos, eles caminhavam pela cidade e exploravam novos caminhos. O cachorro aprendeu que a vida é bela quando temos alguém que nos ama.",
        "Na cidade grande, as pessoas corriam para todo lado. O trânsito era intenso e barulhento. Mas no parque, tudo era calma e tranquilo. As crianças brincavam na praça. Os velhos sentavam no banco e observavam os pássaros. O sol brilhava e a brisa sopava suavemente. Era uma tarde perfeita na cidade.",
        "O professor explicava a lição com paciência. Os estudantes ouviam com atenção. A aula era sobre a natureza e os animais. A menina fez uma pergunta interessante. O professor respondeu com alegria. A turma aprendeu muito naquele dia. O conhecimento é uma luz que ilumina o caminho.",
        "A cozinheira preparava um jantar delicioso. Ela cozinhava com muito cuidado. O jantar tinha arroz, feijão e salada. A família comeu com alegria. Todos elogiaram a comida da cozinheira. O segredo dela era colocar amor em cada prato. A cozinha era o coração daquela casa.",
        "O artista pintava um quadro muito bonito. Ele usava cores vivas e fortes. O quadro mostrava uma paisagem da montanha. As pessoas admiravam a pintura na galeria. O artista era feliz com o seu trabalho. Ele dizia que a arte é a janela da alma. Cada pincelada contava uma história.",
        "A cientista trabalhava no laboratório. Ela estudava as estrelas e os planetas. A cientista descobriu uma nova estrela. A descoberta era muito importante. O mundo inteiro celebrava a descoberta da cientista. Ela provou que a curiosidade é o motor da ciência. Um novo capítulo da astronomia começava.",
        "No jardim da casa, as flores cresciam com beleza. O sol brilhava e a chuva caía com cuidado. As borboletas voavam entre as flores. O gato dormia na grama verde. Era um dia tranquilo e bonito. A natureza mostra que a simplicidade é a maior riqueza.",
        "O menino e a menina brincavam na praia. Eles construíam castelos de areia. O mar era azul e calmo. Os pássaros voavam sobre a água. A tarde era quente e ensolarada. As crianças estavam felizes. A praia é um lugar mágico onde sonhos se realizam.",
        "A música enchia a sala de concerto. O violinista tocava com paixão. Cada nota era uma emoção profunda. A plateia segurava a respiração. No final, a oviação foi enorme. A música tem o poder de tocar almas e unir corações.",
        "O navio navegava pelo oceano azul. Os marinheiros trabalhavam com determinação. O vento soprava as velas e o barco avançava. No horizonte, uma ilha apareceu. Era uma ilha misteriosa cheia de árvores e cachoeiras. A tripulação decidiu explorar e encontrou um tesouro escondido.",
        "A chuva caía suavemente sobre o telhado. A mulher sentava na janela com um livro. O som da chuva era relaxante e tranquilo. Ela lia página após página, imersa na história. Quando a chuva parou, um arco-íris apareceu no céu. Era como se a natureza estivesse pintando um quadro.",
        "O avião sobrevoava as montanhas nevadas. Os passageiros olhavam pela janela maravilhados. As nuvens formavam castelos brancos no céu. O piloto anunciou que Logo chegariam ao destino. A viagem era longa, mas a vista compensava cada minuto. A beleza vista de cima é impossível de esquecer.",
        "Na fazenda, o galo cantava ao amanhecer. As vacas pastavam no campo verde. O fazendeiro cuidava dos animais com carinho. A vida no campo era simples mas feliz. O ar era puro e a água cristalina. Tudo na natureza funcionava em harmonia perfeita.",
        "No laboratório da escola, a turma misturou água, sal e corante. A professora explicou que cada experiência precisava de observação e registro. Quando os cristais apareceram no copo, todos ficaram admirados. A ciência parecia uma aventura feita de perguntas.",
        "A floresta acordou com o canto dos pássaros. A borboleta passou pelas flores amarelas e o coelho saiu devagar da toca. Perto do lago, a tartaruga tomou sol em silêncio. A manhã mostrou que cada ser vivo tinha seu ritmo.",
        "Na avenida movimentada, o carteiro entregava cartas enquanto a jornalista anotava notícias. Os ônibus paravam na estação e as pessoas caminhavam com pressa. Mesmo assim, um músico tocava violão na esquina. A cidade tinha barulho, trabalho e poesia.",
        "O time treinou no estádio antes do campeonato. A treinadora pediu atenção, respeito e coragem. No último minuto, a menina marcou um gol bonito. A vitória foi celebrada como fruto da união.",
        "A banda ensaiava numa garagem pequena. O baterista marcava o ritmo e a cantora procurava a nota certa. Depois de muitas tentativas, a música finalmente ganhou forma. Os vizinhos aplaudiram pela janela.",
        "Na cozinha da avó, o cheiro de pão quente enchia a casa. O padeiro ensinou a sovar a massa com paciência. As crianças esperaram o forno apitar. Quando o pão ficou pronto, todos comeram com manteiga e alegria.",
        "A família viajou de trem até uma cidade antiga. Pela janela, apareciam rios, plantações e pontes de pedra. No museu, aprenderam histórias sobre outros tempos. A viagem deixou lembranças luminosas.",
        "Duas amigas encontraram uma carteira perdida no parque. Elas procuraram um guarda e entregaram tudo com cuidado. O dono agradeceu emocionado. Naquele dia, as meninas entenderam que honestidade também é amizade.",
        "O inverno chegou com noites frias e cobertores pesados. A família preparou sopa e chocolate quente. Do lado de fora, a chuva batia no telhado. Dentro de casa, as conversas aqueciam o coração.",
        "Na primavera, a florista abriu a loja bem cedo. Rosas, lírios e margaridas coloriam as mesas. Um menino comprou uma flor para a mãe. O pequeno presente deixou a manhã mais doce.",
        "O golfinho nadava perto do barco dos pesquisadores. A equipe anotava sons, movimentos e caminhos no mar. Ninguém tocava no animal, apenas observava com respeito. A natureza ensinava sem precisar falar.",
        "Na escola, a biblioteca ganhou novos livros. O professor organizou uma roda de leitura no pátio. Cada estudante escolheu uma história diferente. Ao final, todos queriam contar o que haviam imaginado.",
        "No hospital, a enfermeira caminhava pelos corredores com calma. Ela conversava com os pacientes e lembrava os horários dos remédios. O médico explicava cada exame com clareza. O cuidado fazia o medo diminuir.",
        "Durante as festas de junho, a praça recebeu bandeirinhas coloridas. Havia milho, música e dança ao redor da fogueira. A comunidade inteira ajudou na organização. A noite terminou com risadas e céu estrelado.",
        "O menino sentia saudade do avô que morava longe. Ele escreveu uma carta contando sobre a escola e o cachorro. Dias depois, recebeu uma resposta cheia de carinho. A saudade ficou menor quando as palavras chegaram.",
        "A arquiteta desenhou uma casa com janelas grandes e jardim aberto. Ela pensou na luz da manhã e na sombra da tarde. A família acompanhou cada detalhe do projeto. Quando a obra terminou, a casa parecia abraçar quem entrava.",
        "Na praia limpa, voluntários recolheram plástico e redes antigas. A tartaruga voltou ao mar sem obstáculos. Crianças aprenderam por que o lixo machuca os animais. Cuidar do oceano virou promessa coletiva.",
        "O mecânico abriu a oficina antes do nascer do sol. Ele ouviu o motor do carro e encontrou o problema. Com ferramentas simples, consertou a peça quebrada. O motorista seguiu viagem agradecido.",
        "A psicóloga organizou uma conversa sobre emoções na escola. Os estudantes falaram de medo, raiva, alegria e vergonha. Ninguém riu das histórias dos colegas. A escuta transformou a sala num lugar mais seguro.",
        "No acampamento, as crianças aprenderam a montar barracas. À noite, observaram estrelas e inventaram constelações. O vento balançava as árvores com suavidade. Dormiram felizes depois de uma aventura simples.",
    ]

    frases.extend(historias * 3)

    # Frases soltas — mais variadas
    frases_soltas = [
        "O sol brilha no céu azul.",
        "A chuva cai na cidade.",
        "O vento sopra na montanha.",
        "A lua ilumina a noite.",
        "As estrelas brilham no escuro.",
        "O rio corre para o mar.",
        "A floresta é verde e bonita.",
        "O fogo esquenta a casa.",
        "A água é importante para a vida.",
        "A natureza é maravilhosa.",
        "O dia amanheceu bonito.",
        "O pôr do sol era lindo.",
        "A noite estava estrelada.",
        "A manhã chegou com alegria.",
        "A tarde foi tranquila.",
        "O aprendizado nunca termina.",
        "A leitura abre portas.",
        "O conhecimento transforma vidas.",
        "A amizade é um tesouro.",
        "O amor é a maior força.",
        "A vida é bela e curta.",
        "O tempo passa depressa.",
        "A esperança nunca morre.",
        "O mundo é cheio de surpresas.",
        "A música alegra o coração.",
        "A sabedoria vem com os anos.",
        "A paciência é uma virtude.",
        "O respeito é fundamental.",
        "A diversidade nos enriquece.",
        "A criatividade não tem limites.",
        "A simpler explicaçao é frequentemente a melhor.",
        "Todo grande jornada começa com um passo.",
        "A prática leva à perfeição.",
        "Nunca é tarde para aprender.",
        "A coragem não é a ausência de medo.",
        "A imaginação é mais importante que o conhecimento.",
        "A gentileza torna o caminho mais leve.",
        "A curiosidade move a ciência.",
        "O trabalho honesto constrói confiança.",
        "A conversa sincera evita muitos conflitos.",
        "Cada estação tem sua beleza.",
        "O silêncio também pode ensinar.",
        "A família guarda memórias importantes.",
        "O esporte ensina disciplina e cooperação.",
        "Cozinhar é uma forma de cuidado.",
        "Viajar amplia o olhar sobre o mundo.",
    ]

    frases.extend(frases_soltas * 10)

    # Diálogos simples
    dialogos = [
        "— Bom dia! — disse a menina com um sorriso. — Bom dia! — respondeu o menino alegremente.",
        "— O que você está lendo? — perguntou o professor. — Estou lendo um livro sobre ciência! — respondeu a estudante.",
        "— Vamos brincar no parque? — sugeriu o cachorro. — Sim, vamos! — disseram as crianças.",
        "— A comida está deliciosa! — elogiou o visitante. — Obrigada! — disse a cozinheira satisfeita.",
        "— Qual é o seu sonho? — perguntou a amiga. — Eu quero ser cientista! — respondeu a menina.",
        "— Você viu a chuva chegando? — perguntou o carteiro. — Vi, e trouxe um guarda-chuva extra — respondeu a florista.",
        "— O treino foi difícil? — perguntou o pai. — Foi, mas eu melhorei meu tempo — disse a atleta.",
        "— Por que as estrelas brilham? — perguntou a criança. — Porque são bolas enormes de gás quente — explicou a professora.",
        "— Posso ajudar na cozinha? — perguntou o menino. — Pode lavar os legumes com cuidado — respondeu a avó.",
        "— O ônibus já passou? — perguntou a jornalista. — Ainda não, ele chega em cinco minutos — disse o motorista.",
        "— Essa música é nova? — perguntou a vizinha. — Sim, compus ontem à noite — respondeu o músico.",
        "— O paciente está melhor? — perguntou a enfermeira. — Está respirando com mais calma — respondeu o médico.",
        "— Vamos plantar uma árvore? — sugeriu a menina. — Vamos cuidar dela todos os dias — respondeu o amigo.",
        "— O mar está agitado hoje? — perguntou o pescador. — Está, precisamos esperar o vento diminuir — disse o piloto.",
        "— Você terminou o desenho? — perguntou a arquiteta. — Terminei e acrescentei uma janela maior — disse o estudante.",
        "— A feira está cheia? — perguntou a mãe. — Está cheia de frutas maduras — respondeu a filha.",
        "— O livro te emocionou? — perguntou o avô. — Sim, parecia falar comigo — respondeu a neta.",
        "— Podemos observar a borboleta? — perguntou o menino. — Podemos, mas sem tocar nas asas — explicou a veterinária.",
        "— O que faremos nas férias? — perguntou o irmão. — Visitaremos a montanha e o lago — respondeu a irmã.",
        "— Você está triste? — perguntou a amiga. — Estou, mas conversar ajuda bastante — respondeu o colega.",
    ]
    frases.extend(dialogos * 5)

    # Fatos sobre o mundo
    fatos = [
        "A Terra gira em torno do Sol.",
        "A água ferve a cem graus Celsius ao nível do mar.",
        "O Brasil é o maior país da América do Sul.",
        "A Lua é o satélite natural da Terra.",
        "As plantas produzem oxigênio durante a fotossíntese.",
        "O coração bombeia sangue pelo corpo.",
        "Os peixes respiram principalmente por brânquias.",
        "As abelhas ajudam na polinização de muitas plantas.",
        "O arco-íris aparece quando a luz atravessa gotas de água.",
        "O inverno é uma das quatro estações do ano.",
        "O oceano cobre a maior parte da superfície da Terra.",
        "A Amazônia abriga enorme diversidade de seres vivos.",
        "O som precisa de um meio material para se propagar.",
        "A luz viaja mais rápido que o som.",
        "Os livros podem preservar histórias por muitos anos.",
        "A vacinação ajuda a prevenir doenças.",
        "O trânsito fica mais seguro com respeito às regras.",
        "A reciclagem reduz a quantidade de lixo descartado.",
        "O café é uma bebida muito consumida no Brasil.",
        "O futebol é um esporte praticado em muitos países.",
        "A música pode combinar ritmo, melodia e harmonia.",
        "A culinária brasileira mistura influências indígenas, africanas e europeias.",
        "As bibliotecas guardam livros, jornais e outros materiais de consulta.",
        "O calendário organiza dias, semanas, meses e anos.",
        "A bússola ajuda a indicar direções.",
        "O sol fornece luz e calor para a Terra.",
        "Os rios levam água doce por diferentes regiões.",
        "As montanhas podem se formar por movimentos da crosta terrestre.",
        "O corpo humano precisa de água para funcionar bem.",
        "A educação amplia oportunidades e conhecimentos.",
        "As nuvens são formadas por pequenas gotas de água ou cristais de gelo.",
        "Os mapas representam lugares e caminhos.",
        "O mel é produzido pelas abelhas a partir do néctar das flores.",
        "A gravidade atrai os objetos em direção à Terra.",
        "O relógio mede a passagem do tempo.",
    ]
    frases.extend(fatos * 6)

    # Descrições de cenas, objetos e ambientes
    descricoes = [
        "O céu estava azul e sem nuvens.",
        "A casa tinha uma porta vermelha e janelas grandes.",
        "O jardim cheirava a terra molhada e flores recém-abertas.",
        "A rua estreita tinha pedras antigas e postes amarelos.",
        "O mercado estava cheio de frutas coloridas e vozes animadas.",
        "A biblioteca era silenciosa, iluminada por lâmpadas suaves.",
        "O rio passava devagar entre árvores altas e sombras frescas.",
        "A montanha parecia dourada ao receber a luz do fim da tarde.",
        "O quarto era pequeno, mas organizado e cheio de livros.",
        "A cozinha tinha panelas brilhantes e cheiro de bolo assado.",
        "O estádio vibrava com bandeiras, tambores e cantos da torcida.",
        "A praia estava calma, com ondas baixas e areia clara.",
        "A floresta era densa, úmida e repleta de sons escondidos.",
        "O museu tinha corredores amplos e quadros de cores profundas.",
        "A estação parecia apressada, com malas, anúncios e passos rápidos.",
        "O hospital tinha paredes claras e corredores muito limpos.",
        "A fazenda exibia campos verdes, cercas de madeira e um céu imenso.",
        "O escritório tinha mesas alinhadas e plantas perto da janela.",
        "A caverna era fria, escura e cheia de ecos.",
        "O deserto se estendia em ondas de areia sob o sol forte.",
        "A neve cobria telhados, ruas e galhos com silêncio branco.",
        "O porto tinha barcos coloridos, redes secando e cheiro de sal.",
        "A sala de aula estava enfeitada com cartazes e desenhos dos estudantes.",
        "O restaurante tinha luz baixa, música tranquila e mesas bem postas.",
    ]
    frases.extend(descricoes * 8)

    random.seed(42)
    random.shuffle(frases)
    texto = " ".join(frases)
    return texto


# ──────────────────────────────────────────────────────────
# Augmentação de dados
# ──────────────────────────────────────────────────────────

def augmentar_texto(texto: str, prob_dropout: float = 0.1) -> str:
    """
    Augmentação por token dropout: remove caracteres aleatoriamente.

    Simula ruído e força o modelo a aprender representações mais robustas.
    Distribui uniformemente entre os caracteres, evitando caracteres de pontuação.
    """
    random.seed(None)
    resultado = []
    for ch in texto:
        if ch in ".!?,;:" and random.random() < prob_dropout * 0.5:
            continue
        if random.random() < prob_dropout * 0.3:
            continue
        resultado.append(ch)
    return "".join(resultado)


def permutar_frases(texto: str) -> str:
    """Embaralha as frases do corpus para criar variações estruturais."""
    frases = [f.strip() for f in texto.split(".") if f.strip()]
    frases = [f + "." for f in frases]
    random.shuffle(frases)
    return " ".join(frases)


# ──────────────────────────────────────────────────────────
# Dataset SFT (Supervised Fine-Tuning)
# ──────────────────────────────────────────────────────────

def gerar_dados_sft() -> list[tuple[str, str]]:
    """
    Gera pares instrução-resposta para fine-tuning supervisionado.

    Formato: (instrução, resposta)
    Durante o treino, a loss é computada apenas nos tokens da resposta.
    """
    dados = [
        ("O que é o sol?", "O sol é uma estrela que ilumina e aquece a Terra. Ele é essencial para a vida no nosso planeta."),
        ("O que os gatos gostam de fazer?", "Os gatos gostam de dormir, brincar, caçar e receber carinho. São animais curiosos e independentes."),
        ("Como é a praia?", "A praia é um lugar bonito com areia, mar e sol. As pessoas nadam, brincam e relaxam na praia."),
        ("O que é a chuva?", "A chuva é água que cai das nuvens. Ela é importante para as plantas, os rios e a natureza."),
        ("Quem é o professor?", "O professor é uma pessoa que ensina e ajuda os estudantes. Ele compartilha conhecimento com paciência e dedicação."),
        ("O que é a amizade?", "A amizade é um sentimento de afeto e confiança entre pessoas. Amigos se apoiam e compartilham momentos juntos."),
        ("Como é a floresta?", "A floresta é um lugar cheio de árvores, plantas e animais. É verde, fresca e muito importante para o planeta."),
        ("O que é a música?", "A música é uma forma de arte que usa sons organizados de maneira harmoniosa. Ela pode expressar emoções e contar histórias."),
        ("O que você faz na escola?", "Na escola, nós estudamos, aprendemos coisas novas, fazemos amigos e nos preparamos para o futuro."),
        ("O que é a leitura?", "A leitura é o ato de interpretar textos escritos. Ela nos leva a mundos imaginários e nos ensina sobre a vida."),
        ("Como é a cidade?", "A cidade é um lugar com muitas pessoas, prédios, ruas e carros. Tem parques, escolas e muitos lugares para visitar."),
        ("O que é a ciência?", "A ciência é o estudo do mundo natural. Os cientistas fazem experiências para entender como as coisas funcionam."),
        ("Por que o céu é azul?", "O céu parece azul porque a luz do sol se espalha ao passar pela atmosfera. As cores mais azuis se espalham mais."),
        ("O que são as estrelas?", "As estrelas são grandes esferas de gás quente que brilham no céu. O sol é a estrela mais perto da Terra."),
        ("Como é o inverno?", "O inverno é a estação mais fria do ano. Os dias são mais curtos, as pessoas usam roupas quentes e às vezes neva."),
        ("O que é um jardim?", "O jardim é um lugar com flores, plantas e árvores. As pessoas cuidam do jardim regando e podando as plantas."),
        ("Quem é a cozinheira?", "A cozinheira é uma pessoa que prepara comida com habilidade e carinho. Ela cozinha pratos deliciosos para os outros."),
        ("O que é a coragem?", "A coragem é a força de enfrentar o medo e as dificuldades. Coragem não é ausência de medo, mas a decisão de agir apesar dele."),
        ("Como funcionam os sonhos?", "Os sonhos são imagens e histórias que o cérebro cria enquanto dormimos. Eles podem ser bonitos, estranhos ou assustadores."),
        ("O que é a felicidade?", "A felicidade é um sentimento de alegria e contentamento. Cada pessoa encontra a felicidade de um jeito diferente."),
    ]
    return dados


# ──────────────────────────────────────────────────────────
# Dataset DPO (Direct Preference Optimization)
# ──────────────────────────────────────────────────────────

def gerar_dados_dpo() -> list[tuple[str, str, str]]:
    """
    Gera pares de preferência para DPO.

    Formato: (prompt, resposta_escolhida, resposta_rejeitada)
    resposta_escolhida = resposta boa (coerente, correta)
    resposta_rejeitada = resposta ruim (incoerente, errada)
    """
    dados = [
        ("O que é o sol?", "O sol é uma estrela que ilumina e aquece a Terra.", "O sol é uma planeta grande."),
        ("Como é a praia?", "A praia é um lugar bonito com areia e mar.", "A praia é um prédio grande."),
        ("O que os gatos gostam?", "Os gatos gostam de dormir e brincar.", "Os gatos gostam de pilotar aviões."),
        ("O que é a chuva?", "A chuva é água que cai das nuvens.", "A chuva é fogo que cai do céu."),
        ("Quem é o professor?", "O professor é uma pessoa que ensina os estudantes.", "O professor é um animal que vive na floresta."),
        ("O que é a amizade?", "A amizade é um sentimento de carinho e confiança.", "A amizade é uma planta que cresce no jardim."),
        ("O que é a ciência?", "A ciência é o estudo do mundo natural.", "A ciência é um tipo de comida."),
        ("Como é o inverno?", "O inverno é a estação mais fria do ano.", "O inverno é a estação mais quente do ano."),
        ("O que são estrelas?", "As estrelas são esferas de gás que brilham no céu.", "As estrelas são pedras no chão."),
        ("O que é a música?", "A música é uma arte que organiza sons de forma harmoniosa.", "A música é um esporte radical."),
    ]
    return dados


# ──────────────────────────────────────────────────────────
# Carregar/salvar corpus
# ──────────────────────────────────────────────────────────

def salvar_corpus(caminho: str = "data/corpus.txt") -> str:
    """Gera e salva o corpus em arquivo. Retorna o texto."""
    texto = gerar_corpus()
    Path(caminho).write_text(texto, encoding="utf-8")
    print(f"Corpus salvo em '{caminho}': {len(texto):,} caracteres")
    return texto


def carregar_corpus(caminho: str = "data/corpus.txt") -> str:
    """Carrega corpus de arquivo, ou gera se não existir."""
    path = Path(caminho)
    if path.exists():
        texto = path.read_text(encoding="utf-8")
        print(f"Corpus carregado de '{caminho}': {len(texto):,} caracteres")
        return texto
    return salvar_corpus(caminho)


def carregar_corpus_externo(caminho: str) -> str:
    """Carrega corpus de um arquivo de texto externo (ex: OSCAR, brWac)."""
    path = Path(caminho)
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")
    texto = path.read_text(encoding="utf-8")
    print(f"Corpus externo carregado de '{caminho}': {len(texto):,} caracteres")
    return texto
PYEOF

echo "✓ train.py, generate.py, sft.py, dpo.py, data/corpus.py"

In [ ]:
#@title 6. Pré-treinamento { display-mode: "form" }

#@markdown ## Configuração
D_MODEL = 64         #@param {type:"integer"}
N_HEADS = 4          #@param {type:"integer"}
N_LAYERS = 2         #@param {type:"integer"}
CONTEXT_LEN = 256    #@param {type:"integer"}
BATCH_SIZE = 64        #@param {type:"integer"}
MAX_EPOCHS = 30        #@param {type:"integer"}
LEARNING_RATE = 3e-4 #@param {type:"number"}
GRADIENT_ACCUM = 1    #@param {type:"integer"}
TOKENIZER = "bpe"     #@param ["bpe", "char"]
USE_ROPE = True       #@param {type:"boolean"}

import sys
sys.path.insert(0, 'minigpt')
import torch

from config import GPTConfig
from train import treinar
from tokenizer import criar_tokenizer
from data.corpus import gerar_corpus

device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = GPTConfig(
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
    context_len=CONTEXT_LEN, batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS, learning_rate=LEARNING_RATE,
    gradient_accum_steps=GRADIENT_ACCUM,
    tokenizer_type=TOKENIZER, use_rope=USE_ROPE,
)

texto = gerar_corpus()
print(f'Corpus: {len(texto):,} caracteres')

modelo, tokenizer = treinar(config, texto, saida_dir='minigpt/output', device=device)

In [ ]:
#@title 7. SFT — Supervised Fine-Tuning (opcional)
SFT_EPOCHS = 5        #@param {type:"integer"}
SFT_LR = 1e-5         #@param {type:"number"}

from sft import treinar_sft
from model import GPTModel
from data.corpus import gerar_dados_sft
from tokenizer import carregar_tokenizer

# Carregar melhor modelo
checkpoint = torch.load('minigpt/output/melhor_modelo.pt', map_location=device, weights_only=False)
config_sft = checkpoint['config']
if not hasattr(config_sft, 'use_rope'): config_sft.use_rope = True
if not hasattr(config_sft, 'tokenizer_type'): config_sft.tokenizer_type = 'bpe'
if not hasattr(config_sft, 'sft_lr'): config_sft.sft_lr = 1e-5
if not hasattr(config_sft, 'sft_epochs'): config_sft.sft_epochs = 5
if 'position_embedding.weight' in checkpoint['modelo']: config_sft.use_rope = False
tokenizer = carregar_tokenizer('minigpt/output/tokenizer.json')
modelo = GPTModel(config_sft).to(device)
modelo.load_state_dict(checkpoint['modelo'], strict=False)

config_sft.sft_epochs = SFT_EPOCHS
config_sft.sft_lr = SFT_LR
dados = gerar_dados_sft()
print(f'SFT data: {len(dados)} pares')

modelo = treinar_sft(modelo, tokenizer, config_sft, dados, saida_dir='minigpt/output', device=device)

In [ ]:
#@title 8. DPO — Direct Preference Optimization (opcional)
DPO_EPOCHS = 3        #@param {type:"integer"}
DPO_BETA = 0.1        #@param {type:"number"}

from dpo import treinar_dpo
from model import GPTModel
from data.corpus import gerar_dados_dpo
import os

# Carregar modelo (preferir SFT)
modelo_path = 'minigpt/output/melhor_modelo_sft.pt'
if not os.path.exists(modelo_path):
    modelo_path = 'minigpt/output/melhor_modelo.pt'
checkpoint = torch.load(modelo_path, map_location=device, weights_only=False)
config_dpo = checkpoint['config']
if not hasattr(config_dpo, 'use_rope'): config_dpo.use_rope = True
if not hasattr(config_dpo, 'dpo_beta'): config_dpo.dpo_beta = 0.1
if not hasattr(config_dpo, 'dpo_lr'): config_dpo.dpo_lr = 1e-6
if not hasattr(config_dpo, 'dpo_epochs'): config_dpo.dpo_epochs = 3
if 'position_embedding.weight' in checkpoint['modelo']: config_dpo.use_rope = False
tokenizer = carregar_tokenizer('minigpt/output/tokenizer.json')
modelo = GPTModel(config_dpo).to(device)
modelo.load_state_dict(checkpoint['modelo'], strict=False)

config_dpo.dpo_epochs = DPO_EPOCHS
config_dpo.dpo_beta = DPO_BETA
dados = gerar_dados_dpo()
print(f'DPO data: {len(dados)} pares')

modelo = treinar_dpo(modelo, tokenizer, config_dpo, dados, saida_dir='minigpt/output', device=device)

In [ ]:
#@title 9. Testar o modelo
modelo.eval()
modelo_cpu = modelo.to('cpu')

from generate import gerar, gerar_variacoes, gerar_beam_search

print("=== Geração com Top-p + Repetition Penalty ===")
for prompt in ["O gato", "A menina", "Na cidade", "Era uma vez", "Instrução: O que é o sol? Resposta:"]:
    resultado = gerar(modelo_cpu, tokenizer, prompt, max_tokens=120,
                      temperature=0.8, top_k=40, top_p=0.9,
                      repetition_penalty=1.2, device='cpu')
    print(f'Prompt: "{prompt}"')
    print(f'Saída:  {resultado}\n')

print("=== Beam Search ===")
for prompt in ["O gato", "A menina"]:
    resultado = gerar_beam_search(modelo_cpu, tokenizer, prompt,
                                   max_tokens=60, beam_width=5, device='cpu')
    print(f'Prompt: "{prompt}"')
    print(f'Beam:   {resultado}\n')

In [ ]:
#@title 10. Baixar resultados
from google.colab import files
import os

output_dir = 'minigpt/output'
for f in os.listdir(output_dir):
    fpath = os.path.join(output_dir, f)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f'{f}: {size_mb:.1f} MB')

print('\nBaixando arquivos...')
for f in os.listdir(output_dir):
    fpath = os.path.join(output_dir, f)
    files.download(fpath)